# Logistic Regression am Beispiel des Titanic Datensatzes


Am Beispiel des Titanic Datensatzes wollen wir mit Hilfe der Logistischen Regression voraussagen, ob eine Person überlebt hat oder umgekommen ist.
Dazu werden zusätzliche Information wie Geschlecht, Alter, Passagierklasse, Fahrkartenpreis usw. herangezogen.

zusätzlich werden folgende Themen behandelt:
- Umgang mit fehlenden Daten ("missing data"): Imputation
- Erstellen von Dummy-Variablen für Variable mit Klassen und Kategorien


2026-06-16 ug V1.5
 
----
Titanic-Datensatz:
- integriert in Seaborn `sns.load_dataset("titanic")`
- [Download bei kaggle](https://www.kaggle.com/c/titanic/overview)

---
Referenzen:

[Logistic Regression: Titanic Dataset](https://www.youtube.com/watch?v=in6PDZXkqrw)

<!--
Unterdrücken von Warnungen, die durch Seaborn 0.12.2 entstehen, siehe [seaborn - github - issue 3462](https://github.com/mwaskom/seaborn/issues/3462) und [Seaborn futurewarning caused by pandas dataframe](https://stackoverflow.com/questions/77882407/seaborn-futurewarning-caused-by-pandas-dataframe)

import warnings
warnings.filterwarnings("ignore", "is_categorical_dtype")
warnings.filterwarnings("ignore", "use_inf_as_na")
warnings.filterwarnings("ignore", category=FutureWarning, module="seaborn")
-->

### Standard Paket für Datenanalyse laden

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

Anzeigen der verwendeten Seaborn Version:

In [ ]:
sns.__version__

---
### Step1: Titanic Datensatz laden

Seaborn stell den Titanic-Datensatz als Pandas DataFrame zur Verfügung:

In [ ]:
df  = sns.load_dataset("titanic")
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

Spaltenbeschreibungen

| Spaltenname| Erläuterung| Schlüssel |
|---|---|---|
| survived |  Überlebt Status| 0: umgekommen,  1: überlebt|
| pclass   | Passagierklasse | 1: erste Klasse, 2: zweiter Klasse, 3: dritter Klasse|   
| sex      | Geschlecht:| 'male', 'female' |  
| age      | Alter in Jahren| |
| sibsp    | number of siblings/spouses aboard the Titanic|Anzahl Geschwister/Ehepartner|   
| parch    | number of parents / children aboard the Titanic|Anzahl Eltern und Kinder |  
| fare     |Passenger fare | Passagierfahrpreis |
| embarked | Zustiegshafen | 'C':'Cherbourg', 'Q':'Queenstown',  'S':'Southampton'|    | class    | | |  
| who      | | |  
| adult_male| | |    
| deck       | | |
| embark_town| Zustiegshafen im Klartext | |  
| alive      | | |
| alone      | | |  


In [ ]:
df['sex'].unique()

In [ ]:
df['embarked'].unique()

In [ ]:
df['embark_town'].unique()

In [ ]:
df['parch'].unique()

---
### Step2 : Exploratory data analysis EDA

Suche nach fehlenden Daten ("missing data")

In [ ]:
df.isnull().sum()

Darstellung als Heatmap ohne Achsenbeschriftung der y-Achse und ohne Farbskala

In [ ]:
sns.heatmap(df.isnull(), yticklabels=False, cbar=False)

Est gibt also einige fehlende Werte, insbesondere bei den Spalten 'age', 'deck' und 'embark_town'. 
<br>
Diese müssen vor der Modellierung behandelt werden.  

### Untersuchung der Target Variablen "survived"

In [ ]:
df["survived"].value_counts()

In [ ]:
sns.countplot(x="survived", data=df);

Unterteiling nach Geschlecht

In [ ]:
df['sex'].value_counts()

In [ ]:
sns.countplot(x="survived", hue="sex", data=df)

In [ ]:
df['pclass'].value_counts()

In [ ]:
sns.countplot(x="survived", hue="pclass", data=df)

Alterverteilung

In [ ]:
sns.displot(df['age'].dropna())

Anzahl Geschwister/Ehepartner - ('sibsp' - number of siblings/spouses aboard the Titanic)

In [ ]:
df['sibsp'].value_counts()

In [ ]:
sns.countplot(x="sibsp", data=df)

Passagierfahrpreis - ('fare' - Passenger fare)

In [ ]:
df['fare'].hist(bins=40);

---
### Step3: Datensatz bereinigen ("Data cleaning")

#### Fehlende Daten sinnvoll ergänzen

[Wikipedia-Imputation](https://de.wikipedia.org/wiki/Imputation_(Statistik))

In [ ]:
plt.figure(figsize=(12,8))
sns.boxplot(x="pclass", y='age', data=df)

In [ ]:
df.groupby("pclass")['age'].median()

Ersetzen fehlender Altersangabe durch den Mittelwert (Median) des Alters in der entsprechenden Passagierklasse 'pclass':
pclass
1    38.233441
2    29.877630
3    25.140620

In [ ]:
def impute_age_old(cols):
    age = cols[0]
    pclass = cols[1]
    
    if pd.isnull(age):
        if pclass == 1:
            return 37
        elif pclass == 2:
            return  29
        else:
            return 24
    else:
        return age

In [ ]:
# Mapping passend zu deinem Beispiel: Klasse 1 -> 37, 2 -> 29, 3 -> 24
age_by_pclass = {
    1: 37, 2: 29, 3: 24,
    '1': 37, '2': 29, '3': 24
}

def impute_age(row):
    if pd.isna(row['age']):
        return age_by_pclass[row['pclass']]
    return row['age']

# df['age'] = df[['age', 'pclass']].apply(impute_age, axis=1)

Unittest:

In [ ]:
assert impute_age(pd.Series({'age': 25, 'pclass': 1})) == 25
assert impute_age(pd.Series({'age': np.nan, 'pclass': 1})) == 37
assert impute_age(pd.Series({'age': np.nan, 'pclass': 2})) == 29
assert impute_age(pd.Series({'age': np.nan, 'pclass': 3})) == 24
assert impute_age(pd.Series({'age': np.nan, 'pclass': '1'})) == 37

fehlende Daten vorher:

In [ ]:
df.isnull().sum()

Impute anwenden

In [ ]:
df['age'] = df[['age','pclass']].apply(impute_age, axis=1)

fehlende Daten nachher:

In [ ]:
df.isnull().sum()

#### Spalten entfernen, die wir für die logistische Regression nicht verwenden wollen:

In [ ]:
df.drop(['deck', 'embark_town', 'alive', 'alone','adult_male','who','class'], axis=1, inplace=True)

In [ ]:
df.head()

wo haben wir jetzt noch fehlende Werte?

In [ ]:
df.isnull().sum()

Wir entfernen jetzt Zeilen, die fehlende Werte enthalten:

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.isnull().sum()

Jetzt haben wir keine fehlenden Werte mehr.

---
### Step 4: Umwandlung von Variablen mit Kategorien und Klassen (Nominalskala, Ordinalskala)

In [ ]:
df.info()

In [ ]:
df[['sex','embarked']].head()

#### Pandas [`.get_dummies()`](https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html)-Funktion

Idee: Wir legen für jede Klasse eine eigene neue Spalte an, in die wir eine “1” schreiben, wenn die Klasse vorliegt, ansonsten schreiben wir eine „0“ hineingeschrieben. Dies wirkt wie eine Art Schalter.

Jedoch dürfen wir bei Anpassen eines Modells an die Daten, keine Spalten verwenden die linear voneinander abhängig sind bzw. eine starke Korrelation miteinander haben. Ansonst wird die Schätzung der Parameter numerisch instabil.
Problem der Multikollinearität [Wikipedia-Multikollinearität](https://de.wikipedia.org/wiki/Multikollinearit%C3%A4t)

Maßnahme: Wenn eine unabhängige Variable n Klassen beinhalten, dürfen wir nur (n-1) Spalten im Modell berücksichtigen.

Mit dem Parameter `drop_first=True` wird die erste Kategorie weg gelassen.



In [ ]:
pd.get_dummies(df['sex']).head()

In [ ]:
pd.get_dummies(df['embarked']).head()

In [ ]:
sex = pd.get_dummies(df['sex'], drop_first=True)
embark = pd.get_dummies(df['embarked'],drop_first=True)

In [ ]:
sex.head()

In [ ]:
embark.head()

In [ ]:
df.drop(['sex','embarked'], axis=1, inplace=True)

In [ ]:
df = pd.concat([df, sex, embark  ],axis=1)
df.head()

---
### Step5: Logistic Regression

Festlegen der abhängigen Varibalen `y` und der unabhängigen Variablen `X`:

In [ ]:
y=df['survived']
X=df.drop(['survived'], axis=1)

Aufteilung in Traings- und Testdatensatz:

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.33, random_state=42)

Erzeugen eines Logistic Regression Objekts:

Wahl eines anderen Lösungsalgorithmus.

In [ ]:
from sklearn.linear_model import LogisticRegression
LogRegModel = LogisticRegression(solver='liblinear')

Modell an die Trainingsdaten anpassen

In [ ]:
LogRegModel.fit(X_train, y_train) 

Voraussage der abhängigen Variablen mit dem Test-Datensatz machen:

In [ ]:
prediction = LogRegModel.predict(X_test)

#### Qualität der Voraussage bewerten

Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, prediction)

Classification Report

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, prediction,target_names=['died','survived']))


---
Tuning von Hyperparametern

In [ ]:
LogRegModel = LogisticRegression(solver='liblinear',C=100)

LogRegModel.fit(X_train, y_train) 
prediction = LogRegModel.predict(X_test)
print(classification_report(y_test, prediction))

bringt nicht so viel.
Wie könnte es weiter gehen:
Zusätzliche Features nutzen:
Im Titanic Datensatz bei [kaggle](https://www.kaggle.com/c/titanic/overview) ist auch der Name der Person angegeben.